In [ ]:
!pip install wandb

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import wandb
from torch.utils.data import DataLoader, TensorDataset, Dataset, random_split
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: shantanugupta2004 (shantanugupta2004-own-use) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# Model

In [ ]:
x = torch.linspace(0, 1, 128)
y = torch.linspace(0, 1, 128)
X_grid, Y_grid = torch.meshgrid(x, y, indexing='ij')
coords = torch.stack([X_grid.flatten(), Y_grid.flatten()], dim=1)
coords = 2.0 * coords - 1.0  # important scaling

In [ ]:
class DeepONetDataset(Dataset):
    def __init__(self, X_data, Y_data, coords, n_points=1000):
        self.X_data = torch.tensor(X_data, dtype=torch.float32)
        self.Y_data = torch.tensor(Y_data, dtype=torch.float32)
        self.coords = coords
        self.n_points = n_points

    def __len__(self):
        return len(self.X_data)

    def __getitem__(self, idx):

        branch_input = self.X_data[idx]  # [3,128,128]

        indices = torch.randint(0, 128*128, (self.n_points,))
        trunk_input = self.coords[indices]

        target_field = self.Y_data[idx].reshape(3, -1).permute(1,0)
        target = target_field[indices]

        return branch_input, trunk_input, target


In [ ]:
class FourierFeatures(nn.Module):
    def __init__(self, in_dim, mapping_size=64, scale=10.0):
        super().__init__()
        B = torch.randn(in_dim, mapping_size) * scale
        self.register_buffer("B", B)

    def forward(self, x):
        x_proj = 2 * torch.pi * x @ self.B
        return torch.cat([torch.sin(x_proj), torch.cos(x_proj)], dim=-1)

class BranchNet(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.GELU(),
            nn.AdaptiveAvgPool2d(1)
        )
        self.fc = nn.Linear(64, latent_dim)

    def forward(self, u):
        features = self.encoder(u).squeeze(-1).squeeze(-1)
        return self.fc(features)


class TrunkNet(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.fourier = FourierFeatures(2, mapping_size=64)

        self.net = nn.Sequential(
            nn.Linear(128, 128),
            nn.GELU(),
            nn.Linear(128, 128),
            nn.GELU(),
            nn.Linear(128, latent_dim * 3)
        )

        self.latent_dim = latent_dim

    def forward(self, x):
        x = self.fourier(x)
        out = self.net(x)
        return out.view(-1, 3, self.latent_dim)

class DeepONet(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.branch = BranchNet(latent_dim)
        self.trunk = TrunkNet(latent_dim)
        self.latent_dim = latent_dim

    def forward(self, u, x):
        B, n_pts, _ = x.shape

        branch_out = self.branch(u)  # [B, latent]
        trunk_out = self.trunk(x.view(-1, 2))
        trunk_out = trunk_out.view(B, n_pts, 3, self.latent_dim)

        branch_out = branch_out.unsqueeze(1).unsqueeze(2)

        output = torch.sum(branch_out * trunk_out, dim=-1)
        return output

In [ ]:
def relative_l2(pred, target):
    num = torch.norm(target - pred, dim=(1,2))
    den = torch.norm(target, dim=(1,2))
    return (num / (den + 1e-8)).mean()

# Training

In [ ]:
import os

In [ ]:
data_path = "/content/drive/MyDrive/LDC Dataset"
geometries = ["harmonics", "nurbs", "skelneton"]

for geometry in geometries:

    wandb.init(
        project="DeepONet_LDC_uvp_ff",
        name=f"Improved_DeepONet_{geometry}",
        reinit=True,
        config={
            "epochs": 100,
            "batch_size": 8,
            "lr": 1e-3,
            "latent_dim": 256,
            "n_points": 1000
        }
    )

    # -------------------------
    # Load + Normalize Data
    # -------------------------
    X_data = np.load(os.path.join(data_path, f"{geometry}_lid_driven_cavity_X.npz"))['data']
    Y_data = np.load(os.path.join(data_path, f"{geometry}_lid_driven_cavity_Y.npz"))['data'][:,0:3]

    # Normalize
    X_mean, X_std = X_data.mean(), X_data.std()
    Y_mean, Y_std = Y_data.mean(), Y_data.std()

    X_data = (X_data - X_mean) / (X_std + 1e-8)
    Y_data = (Y_data - Y_mean) / (Y_std + 1e-8)

    dataset = DeepONetDataset(X_data, Y_data, coords, wandb.config.n_points)

    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset,
                              batch_size=wandb.config.batch_size,
                              shuffle=True)

    val_loader = DataLoader(val_dataset,
                            batch_size=wandb.config.batch_size)

    model = DeepONet(wandb.config.latent_dim).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=wandb.config.lr)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=wandb.config.epochs
    )

    loss_fn = nn.MSELoss()

    # -------------------------
    # Training
    # -------------------------
    for epoch in range(wandb.config.epochs):

        model.train()
        train_loss = 0

        for branch_input, trunk_input, target in train_loader:

            branch_input = branch_input.to(device)
            trunk_input = trunk_input.to(device)
            target = target.to(device)

            optimizer.zero_grad()
            pred = model(branch_input, trunk_input)

            loss = loss_fn(pred, target)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        scheduler.step()
        train_loss /= len(train_loader)

        # -------------------------
        # Validation
        # -------------------------
        model.eval()
        val_loss = 0
        val_l2 = 0

        with torch.no_grad():
            for branch_input, trunk_input, target in val_loader:

                branch_input = branch_input.to(device)
                trunk_input = trunk_input.to(device)
                target = target.to(device)

                pred = model(branch_input, trunk_input)

                val_loss += loss_fn(pred, target).item()
                val_l2 += relative_l2(pred, target).item()

        val_loss /= len(val_loader)
        val_l2 /= len(val_loader)

        wandb.log({
            "Epoch": epoch+1,
            "Train Loss": train_loss,
            "Val Loss": val_loss,
            "Val Relative L2": val_l2,
            "LR": scheduler.get_last_lr()[0]
        })

        print(f"Epoch {epoch+1} | Train {train_loss:.6f} | "
              f"Val {val_loss:.6f} | L2 {val_l2:.6f}")

    wandb.finish()

Epoch 1 | Train 0.951785 | Val 1.221071 | L2 0.975578
Epoch 2 | Train 0.937063 | Val 1.222427 | L2 0.969523
Epoch 3 | Train 0.937898 | Val 1.218167 | L2 1.029118
Epoch 4 | Train 0.938628 | Val 1.218422 | L2 0.945509
Epoch 5 | Train 0.934850 | Val 1.223368 | L2 0.940151
Epoch 6 | Train 0.936421 | Val 1.240255 | L2 0.939655
Epoch 7 | Train 0.937128 | Val 1.219767 | L2 0.941198
Epoch 8 | Train 0.935522 | Val 1.216319 | L2 0.983701
Epoch 9 | Train 0.933382 | Val 1.220197 | L2 0.928620
Epoch 10 | Train 0.934966 | Val 1.216390 | L2 0.931671
Epoch 11 | Train 0.932272 | Val 1.219009 | L2 0.970784
Epoch 12 | Train 0.936750 | Val 1.347042 | L2 1.629743
Epoch 13 | Train 0.936220 | Val 1.215959 | L2 0.936604
Epoch 14 | Train 0.933875 | Val 1.217016 | L2 0.944738
Epoch 15 | Train 0.936090 | Val 1.217109 | L2 0.934016
Epoch 16 | Train 0.935901 | Val 1.216160 | L2 0.979264
Epoch 17 | Train 0.933208 | Val 1.219814 | L2 0.940520
Epoch 18 | Train 0.934108 | Val 1.214477 | L2 0.943490
Epoch 19 | Train 0.

Epoch,▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇███
LR,███████▇▇▇▇▇▇▇▆▆▆▆▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁
Train Loss,██▇▆▆▅▄▄▃▆▃▄▅▃▄▄▄▃▄▄▃▃▄▃▃▃▂▂▁▂▂▃▁▂▂▂▂▂▂▁
Val Loss,▂▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Val Relative L2,▇▄▄█▃▄█▁▄▂▂▃▂▄▂▂▂▂▂▁▁▃▃▂▂▂▂▂▂▂▂▂▂▂▁▂▂▂▂▂
Epoch,100
LR,0
Train Loss,0.92636
Val Loss,1.21428
Val Relative L2,0.91905


Epoch 1 | Train 0.955416 | Val 1.216365 | L2 1.000786
Epoch 2 | Train 0.956580 | Val 1.207240 | L2 1.000603
Epoch 3 | Train 0.953696 | Val 1.203139 | L2 1.010888
Epoch 4 | Train 0.949120 | Val 1.212321 | L2 0.995927
Epoch 5 | Train 0.953859 | Val 1.208303 | L2 0.993725
Epoch 6 | Train 0.949383 | Val 1.214098 | L2 0.992839
Epoch 7 | Train 0.947703 | Val 1.209626 | L2 0.993960
Epoch 8 | Train 0.950731 | Val 1.201792 | L2 0.987888
Epoch 9 | Train 0.947414 | Val 1.208543 | L2 0.980999
Epoch 10 | Train 0.948150 | Val 1.208798 | L2 1.038651
Epoch 11 | Train 0.951760 | Val 1.206458 | L2 0.997847
Epoch 12 | Train 0.945895 | Val 1.211013 | L2 0.977291
Epoch 13 | Train 0.946941 | Val 1.209333 | L2 0.981115
Epoch 14 | Train 0.946622 | Val 1.210315 | L2 0.985400
Epoch 15 | Train 0.946490 | Val 1.208499 | L2 0.975076
Epoch 16 | Train 0.949985 | Val 1.209114 | L2 0.980804
Epoch 17 | Train 0.946400 | Val 1.205720 | L2 0.977442
Epoch 18 | Train 0.945399 | Val 1.201287 | L2 0.979410
Epoch 19 | Train 0.

Epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▆▆▆▆▇▇▇▇▇▇▇███
LR,█████████▇▇▇▇▇▇▆▆▆▅▅▅▅▅▄▄▄▄▃▃▃▂▂▂▂▂▁▁▁▁▁
Train Loss,██▇▅▅▄▄▄▄▄▄▅▃▃▄▄▄▄▄▃▃▃▃▃▂▄▃▃▃▂▃▃▂▃▂▃▁▃▂▂
Val Loss,▄▂▆▄▇▁▄▃▄▄▃▁▂▅▄▄▄▂▅▄▆▄▆▅▆▄█▅▆▄▄▆▇▇▅▆▆▆▅▂
Val Relative L2,▆█▅▅▅▂▁▁▂▁▁▃▄▃▂▁▁▁▅▆▁▄▂▁▂▁▂▄▃▄▃▃▃▂▃▃▃▃▃▃
Epoch,100
LR,0
Train Loss,0.94024
Val Loss,1.20932
Val Relative L2,0.98423


Epoch 1 | Train 0.812353 | Val 1.825818 | L2 1.518179
Epoch 2 | Train 0.803414 | Val 1.777565 | L2 1.565273
Epoch 3 | Train 0.812090 | Val 1.809272 | L2 2.065720
Epoch 4 | Train 0.797301 | Val 1.734221 | L2 1.332263
Epoch 5 | Train 0.820243 | Val 1.771901 | L2 1.080408
Epoch 6 | Train 0.806002 | Val 1.764862 | L2 0.997220
Epoch 7 | Train 0.804633 | Val 1.835875 | L2 2.939193
Epoch 8 | Train 0.806172 | Val 1.780576 | L2 1.597873
Epoch 9 | Train 0.798745 | Val 1.800181 | L2 1.057347
Epoch 10 | Train 0.802002 | Val 1.777406 | L2 1.460635
Epoch 11 | Train 0.805500 | Val 1.796732 | L2 0.995939
Epoch 12 | Train 0.803703 | Val 1.744938 | L2 0.962382
Epoch 13 | Train 0.811725 | Val 1.789564 | L2 1.113560
Epoch 14 | Train 0.815227 | Val 1.818471 | L2 1.109927
Epoch 15 | Train 0.803807 | Val 1.770493 | L2 1.582061
Epoch 16 | Train 0.799615 | Val 1.815571 | L2 1.083407
Epoch 17 | Train 0.802446 | Val 1.768931 | L2 1.074228
Epoch 18 | Train 0.799213 | Val 1.770381 | L2 1.190536
Epoch 19 | Train 0.

Epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▇▇▇▇▇▇██
LR,███████▇▇▇▇▆▆▆▆▅▅▅▅▅▄▄▄▃▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁
Train Loss,▆▂█▄▃▆▇▄▄▆▂▅▇▆▆▃▄▁▆▄▄▃▇▄▄▄▃▄▂▃▆▃▄▃▆▃▄▄▃▃
Val Loss,█▃▆▂▆▅▃▃▃▄▆▂▆▅██▃▅▄▆▄▃▅▅▆▂▄▆▅▃▂▅▅▂▅▄▆▁▃▆
Val Relative L2,▃▃▂▁█▃▁▂▂▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Epoch,100
LR,0
Train Loss,0.79969
Val Loss,1.80101
Val Relative L2,0.92505
